# 🚀 Hướng Dẫn Chạy CrowdDiff Trên Google Colab

Chào mừng bạn đến với notebook chạy dự án **CrowdDiff**. Do hạn chế về phần cứng cục bộ, notebook này sẽ giúp bạn tận dụng GPU miễn phí của Google Colab (như T4 GPU) để khởi chạy, huấn luyện và suy diễn (inference).

**Yêu cầu trước khi bắt đầu:**
1. Bạn đã tải file `crowddiff_colab.zip` từ máy tính cục bộ của bạn.
2. Bạn đã upload file `crowddiff_colab.zip` lên **Google Drive** của bạn (thư mục gốc `My Drive`).

### Bước 1: Kiểm tra phần cứng GPU
Đảm bảo bạn đang sử dụng GPU. Nếu thấy lỗi, hãy vào `Runtime` -> `Change runtime type` và chọn `T4 GPU`.

In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


### Bước 2: Kết nối với Google Drive
Chạy đoạn code dưới đây và cấp quyền để Colab có thể tải file `crowddiff_colab.zip` từ Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 3: Giải nén mã nguồn dự án

In [ ]:
!mkdir -p /content/crowddiff-main
!unzip -q /content/drive/MyDrive/crowddiff_colab.zip -d /content/crowddiff-main
print("Giải nén thành công!")

### Bước 4: Di chuyển vào thư mục dự án và cài đặt thư viện

In [ ]:
iprint('Cài đặt các thư viện hệ thống cần thiết cho lxml...')
!apt-get update && apt-get install -y libxml2-dev libxslt-dev

In [ ]:
print('Cài đặt lại thư viện Python...')
import os
os.chdir('/content/crowddiff-main')

# 1. Nâng cấp công cụ build
!pip install --upgrade pip setuptools wheel

# 2. Sửa file requirements.txt triệt để hơn
!sed -i 's/ @ file.*//g' requirements.txt
!sed -i 's/numpy==1.24.0/numpy>=1.26.0/g' requirements.txt
!sed -i 's/pandas==1.5.2/pandas>=2.0.0/g' requirements.txt
!sed -i 's/grpcio==1.51.1/grpcio>=1.60.0/g' requirements.txt
!sed -i '/pathtools/d' requirements.txt
!sed -i '/brotlipy/d' requirements.txt
!sed -i 's/contourpy==1.0.6/contourpy>=1.2.0/g' requirements.txt

# 3. Cài đặt các thư viện phụ trợ
print('Đang cài đặt các thư viện chính...')
!pip install watchdog kagglehub urllib3==1.26.15

# 4. Cài đặt chính với cờ ưu tiên binary để tránh lỗi build từ source
!pip install -r requirements.txt --only-binary=:all: --prefer-binary || pip install -r requirements.txt

print('\n--- Hoàn tất cài đặt! ---')

### Bước 5: Cấp quyền thực thi và chạy Pipeline hoàn chỉnh!
Lưu ý: Môi trường Colab dùng lệnh python hoặc python3 (không dùng .venv). Script `run_pipeline.sh` đã được thiết kế mở, bạn có thể gọi thủ công nếu script không tương thích hoàn toàn.

In [ ]:
!chmod +x run_pipeline.sh

# Do trong Colab chúng ta sử dụng thẳng môi trường gốc (không dùng .venv)
# nên ta thay thế '.venv/bin/python' thành 'python3' ở trong file bash
!sed -i 's/PYTHON_CMD=\.venv\/bin\/python/PYTHON_CMD=python3/g' run_pipeline.sh
!sed -i 's/\.venv\/bin\/python/python3/g' run_pipeline.sh

# Bắt đầu bước 1: Download & Preprocess (sẽ mất khoảng vài phút)
!bash run_pipeline.sh --steps download,preprocess

### Bước 6: Huấn Luyện (Training)
Việc huấn luyện có thể mất hàng giờ. Hãy đảm bảo giữ tab trình duyệt luôn mở, hoặc tương tác thỉnh thoảng để Colab không tự động ngắt kết nối.

In [ ]:
!bash run_pipeline.sh --steps train

### Bước 7: Lưu trữ Checkpoints (Quan trọng!)
Colab sẽ xóa toàn bộ dữ liệu khi bạn đóng tab. Do đó, bạn cần sao chép weights đã train xong (trong thư mục `results/`) về Google Drive của mình.

In [ ]:
# Code ví dụ sao chép thư mục results sang google drive
!mkdir -p /content/drive/MyDrive/crowddiff_results
!cp -r /content/crowddiff-main/results/* /content/drive/MyDrive/crowddiff_results/
print("Đã sao chép kết quả huấn luyện lên Google Drive!")